In [3]:
from typing import Optional, List
from sqlmodel import SQLModel, Field, Relationship, create_engine

# Tạo bổ sung database cho phần ngữ pháp

In [4]:
class Topic(SQLModel, table=True):
    id: int| None = Field(default=None, primary_key=True)
    name: str
    description: str | None

    lessons: list["Lesson"] = Relationship(back_populates="topic")


# =========================
# Lesson
# =========================
class Lesson(SQLModel, table=True):
    id: int| None = Field(default=None, primary_key=True)
    name: str
    description: str | None

    topic_id: int| None = Field(default=None, foreign_key="topic.id")
    topic: Topic|None = Relationship(back_populates="lessons")

    concepts: list["Concept"] = Relationship(back_populates="lesson")
    exercises: list["Exercise"] = Relationship(back_populates="lesson")


# =========================
# Concept (Core node)
# =========================
class Concept(SQLModel, table=True):
    id: int| None = Field(default=None, primary_key=True)
    name: str
    type: str  # grammar / pattern / word / usage / signal / rule
    description: str | None 
    lesson_id: int = Field(default=None, foreign_key="lesson.id")
    lesson: Lesson|None = Relationship(back_populates="concepts")
    outgoing_relations: list["ConceptRelation"] = Relationship(
        back_populates="from_concept",
        sa_relationship_kwargs={"foreign_keys": "[ConceptRelation.from_concept_id]"},
    )

    incoming_relations: list["ConceptRelation"] = Relationship(
        back_populates="to_concept",
        sa_relationship_kwargs={"foreign_keys": "[ConceptRelation.to_concept_id]"},
    )

    # is_line_break: Optional[bool] = None  # for formatting purposes
    examples: list["Example"] = Relationship(back_populates="concept")


# =========================
# Concept Relation (Graph)
# =========================
class ConceptRelation(SQLModel, table=True):
    id: int| None = Field(default=None, primary_key=True)

    from_concept_id: int = Field(foreign_key="concept.id")
    to_concept_id: int = Field(foreign_key="concept.id")

    relation_type: str | None  # uses, used_for, has_structure, similar_to...

    from_concept: Optional[Concept] = Relationship(
        back_populates="outgoing_relations",
        sa_relationship_kwargs={"foreign_keys": "[ConceptRelation.from_concept_id]"},
    )

    to_concept: Optional[Concept] = Relationship(
        back_populates="incoming_relations",
        sa_relationship_kwargs={"foreign_keys": "[ConceptRelation.to_concept_id]"},
    )


# =========================
# Exercise
# =========================
class Exercise(SQLModel, table=True):
    id: int |None = Field(default=None, primary_key=True)
    name: str

    lesson_id: int = Field(default=None, foreign_key="lesson.id")
    lesson: Lesson|None = Relationship(back_populates="exercises")

    questions: list["Question"] = Relationship(back_populates="exercise")


# =========================
# Question
# =========================
class Question(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)

    content: str | None = None
    question: str | None = None
    answer: str | None = None
    correct_answer: str | None = None

    type: str | None = None
    score: float | None = None
    difficulty: float = 0.0
    exercise_id: int  = Field(default=None, foreign_key="exercise.id")
    exercise: Exercise |None = Relationship(back_populates="questions")


# =========================
# Example (sentence)
# =========================
class Example(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    sentence: str
    explanation: str | None = None
    concept_id: int | None = Field(default=None, foreign_key="concept.id")
    concept: Concept | None = Relationship(back_populates="examples")




# =========================
# Create DB
# =========================
DATABASE_URL = "sqlite:///D:/Nam4-hk2/KhoaLuanTotNghiep/lisenare/lisenare.db"

engine = create_engine(DATABASE_URL, echo=False)


def create_db():
    SQLModel.metadata.create_all(engine)


if __name__ == "__main__":
    create_db()


# Thêm data từ knowledge_graph sang lisenare database ngữ pháp sang database chính (có thể dùng attach để dùng dùng data từ knowledge_graph)

In [ ]:
from sqlmodel import Session, create_engine, select

engine_new = create_engine("sqlite:///D:/Nam4-hk2/KhoaLuanTotNghiep/lisenare/lisenare.db", echo=False)
engine_old = create_engine("sqlite:///D:/Nam4-hk2/KhoaLuanTotNghiep/NguPhap/database_graph/knowledge_graph.db", echo=False)
dict_mapping = {
    "topic": Topic,
    "lesson": Lesson,
    "exercise": Exercise,
    "question": Question,


}
with Session(engine_new) as session_new, Session(engine_old) as session_old:
    # Kiểm tra kết nối đến cả hai cơ sở dữ liệu
    lesson_concepts = session_old.exec(select(Question)).all()
    try:
        for table_name, model in dict_mapping.items():
             records = session_old.exec(select(model)).all()
             for record in records:
                 data = record.model_dump()
                 print(data)
                 new_record = model(**data)
                 session_new.add(new_record)
        session_new.commit()
    except Exception as e:
        session_new.rollback()
        raise e

{'description': None, 'id': 1, 'name': 'CHUYÊN ĐỀ 1: THÌ VÀ SỰ PHỐI THÌ'}
{'description': None, 'id': 2, 'name': 'CHUYÊN ĐỀ 2: CÁC LOẠI ĐỘNG TỪ'}
{'description': None, 'id': 3, 'name': 'CHUYÊN ĐỀ 3: DANH ĐỘNG TỪ VÀ ĐỘNG TỪ NGUYÊN MẪU'}
{'description': None, 'id': 4, 'name': 'CHUYÊN ĐỀ 4: ĐỘNG TỪ KHIẾM KHUYẾT'}
{'description': None, 'id': 5, 'name': 'CHUYÊN ĐỀ 5: CÂU CHỦ ĐỘNG, CÂU BỊ ĐỘNG'}
{'description': None, 'id': 6, 'name': 'CHUYÊN ĐỀ 6: CÂU TRỰC TIẾP, GIÁN TIẾP'}
{'description': None, 'id': 7, 'name': 'CHUYÊN ĐỀ 7: CÂU ĐIỀU KIỆN'}
{'description': None, 'id': 8, 'name': 'CHUYÊN ĐỀ 8: CÂU HỎI ĐUÔI'}
{'description': None, 'id': 9, 'name': 'CHUYÊN ĐỀ 9: MẠO TỪ'}
{'description': None, 'id': 10, 'name': 'CHUYÊN ĐỀ 10: CẤU TẠO TỪ VÀ TỪ LOẠI'}
{'description': None, 'id': 11, 'name': 'CHUYÊN ĐỀ 11: CẤP SO SÁNH'}
{'description': None, 'id': 12, 'name': 'CHUYÊN ĐỀ 12: ĐẠI TỪ VÀ LƯỢNG TỪ'}
{'description': None, 'id': 13, 'name': 'CHUYÊN ĐỀ 13: PHÂN TỪ'}
{'description': None, 'id': 14, 'name':